# Storm Damage Predictor — SHAP Explainability
### Notebook 04: Compute + Save SHAP Values for FastAPI

**Run in:** Google Colab (T4 GPU) — same session as model training, OR fresh session

**What this notebook does:**
1. Load trained classifier + featured parquet
2. Compute SHAP values (TreeExplainer — fast, exact for XGBoost)
3. Plot global feature importance (bar chart)
4. Plot summary beeswarm
5. Show per-class SHAP breakdown
6. Save `shap_explainer.pkl` for FastAPI
7. Verify explainer works on a single prediction

**Output:** `models/shap_explainer.pkl`

## CELL 1 — Install + Import

In [ ]:
# Run if fresh Colab session
!pip install xgboost shap -q

In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
import shap
import matplotlib.pyplot as plt

shap.initjs()  # enables JS plots in notebook

print(f"SHAP version: {shap.__version__}")
print("Imports OK")

## CELL 2 — Load Model + Data

**Two paths:**
- Same session as training → model already in memory as `best_model`
- Fresh session → upload `classifier.pkl` + `storm_featured.parquet` via Files tab

In [ ]:
os.makedirs('models', exist_ok=True)

# ── Load classifier ──────────────────────────────────────────────
# Option A: same session (model already trained)
#   classifier = best_model  # just alias it

# Option B: fresh session (upload classifier.pkl first)
classifier = joblib.load('models/classifier.pkl')
print("Classifier loaded")

# ── Load featured parquet ─────────────────────────────────────────
# Upload storm_featured.parquet via Files tab if fresh session
df = pd.read_parquet('storm_featured.parquet')
print(f"Data shape: {df.shape}")
print(f"Columns: {list(df.columns)}")

## CELL 3 — Reconstruct Feature Matrix

Must match EXACT columns used during training. No damage targets as features.

In [ ]:
# These are the 12 features used during training
# Matches 03_model_training.ipynb exactly
FEATURE_COLS = [
    'STATE',
    'MONTH_NAME',
    'EVENT_TYPE',
    'INJURIES_DIRECT',
    'INJURIES_INDIRECT',
    'DEATHS_DIRECT',
    'DEATHS_INDIRECT',
    'MAGNITUDE',
    'duration_min',
    'season',
    'state_avg_damage',
    'event_avg_damage',
]
TARGET_COL = 'damage_tier'

X = df[FEATURE_COLS]
y = df[TARGET_COL]

print(f"X shape: {X.shape}")
print(f"y distribution:\n{y.value_counts().sort_index()}")

## CELL 4 — Build SHAP Explainer

**Why TreeExplainer:**
- Exact (not approximate) for tree-based models
- 100x faster than KernelExplainer on XGBoost
- Returns SHAP values per class (shape: [n_samples, n_features, n_classes])

In [ ]:
print("Building TreeExplainer...")
explainer = shap.TreeExplainer(classifier)
print("Explainer built")
print(f"Expected value (base rates per class): {explainer.expected_value}")

## CELL 5 — Compute SHAP Values on Sample

1.78M rows is too large to explain all at once. Use 5K sample for global plots.
Per-prediction SHAP (for API) = single row, runs in milliseconds.

In [ ]:
# Sample for global plots — stratified to get all tiers
SAMPLE_SIZE = 5_000

sample_idx = (
    df.groupby(TARGET_COL)
    .apply(lambda g: g.sample(min(len(g), SAMPLE_SIZE // 4), random_state=42))
    .index.get_level_values(1)
)
X_sample = X.loc[sample_idx].reset_index(drop=True)
y_sample = y.loc[sample_idx].reset_index(drop=True)

print(f"Sample shape: {X_sample.shape}")
print(f"Sample tier distribution:\n{y_sample.value_counts().sort_index()}")

print("\nComputing SHAP values (this takes ~1-2 min on T4)...")
shap_values = explainer.shap_values(X_sample)

# shap_values shape: list of 4 arrays (one per class), each [n_samples, n_features]
print(f"SHAP values computed. Type: {type(shap_values)}")
if isinstance(shap_values, list):
    print(f"Shape per class: {shap_values[0].shape}")
    print(f"Number of classes: {len(shap_values)}")
else:
    print(f"Shape: {shap_values.shape}")

## CELL 6 — Global Feature Importance (Bar Chart)

Shows which features matter most ACROSS all classes — mean absolute SHAP.

In [ ]:
TIER_NAMES = ['None', 'Low', 'Medium', 'High']

# Mean absolute SHAP across all classes (global importance)
if isinstance(shap_values, list):
    # Stack: [n_classes, n_samples, n_features] → mean over classes + samples
    shap_array = np.array(shap_values)          # (4, n_samples, n_features)
    mean_abs_shap = np.abs(shap_array).mean(axis=(0, 1))  # (n_features,)
else:
    mean_abs_shap = np.abs(shap_values).mean(axis=0)

importance_df = (
    pd.DataFrame({'feature': FEATURE_COLS, 'importance': mean_abs_shap})
    .sort_values('importance', ascending=False)
    .reset_index(drop=True)
)

print("Global feature importance (mean |SHAP|):")
print(importance_df.to_string(index=False))

# Plot
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(importance_df['feature'][::-1], importance_df['importance'][::-1], color='steelblue')
ax.set_xlabel('Mean |SHAP value|')
ax.set_title('Global Feature Importance — Storm Damage Classifier')
plt.tight_layout()
plt.savefig('models/shap_global_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: models/shap_global_importance.png")

## CELL 7 — Beeswarm Summary Plot

Shows direction of effect: which feature values push toward which damage tier.

In [ ]:
# One beeswarm per class — most informative
if isinstance(shap_values, list):
    for class_idx, tier_name in enumerate(TIER_NAMES):
        plt.figure()
        shap.summary_plot(
            shap_values[class_idx],
            X_sample,
            feature_names=FEATURE_COLS,
            title=f'SHAP Summary — Tier {class_idx}: {tier_name}',
            show=False,
        )
        plt.tight_layout()
        plt.savefig(f'models/shap_summary_tier{class_idx}_{tier_name.lower()}.png',
                    dpi=150, bbox_inches='tight')
        plt.show()
        print(f"Saved: shap_summary_tier{class_idx}_{tier_name.lower()}.png")
else:
    # Multioutput single array (newer SHAP versions)
    shap.summary_plot(shap_values, X_sample, feature_names=FEATURE_COLS)

print("All beeswarm plots done")

## CELL 8 — Per-Prediction SHAP (API Dry Run)

This is what FastAPI will call per request. Test it works on single row.

In [ ]:
def get_shap_for_prediction(
    explainer: shap.TreeExplainer,
    row: pd.DataFrame,
    feature_names: list[str],
    top_n: int = 5,
) -> dict:
    """
    Compute SHAP for single prediction row.
    Returns top-N features driving the predicted class.

    Args:
        explainer: fitted TreeExplainer
        row: single-row DataFrame with model features
        feature_names: list of feature column names
        top_n: how many top features to return

    Returns:
        dict with predicted class, base value, and top features
    """
    shap_vals = explainer.shap_values(row)  # list of 4 arrays, each (1, n_features)

    # Get predicted class
    pred_class = int(classifier.predict(row)[0])

    # SHAP for predicted class only
    if isinstance(shap_vals, list):
        class_shap = shap_vals[pred_class][0]  # (n_features,)
        base_value = float(explainer.expected_value[pred_class])
    else:
        class_shap = shap_vals[0]
        base_value = float(explainer.expected_value)

    # Rank features by absolute SHAP
    top_indices = np.argsort(np.abs(class_shap))[::-1][:top_n]

    top_features = [
        {
            'feature': feature_names[i],
            'shap_value': float(class_shap[i]),
            'feature_value': float(row.iloc[0][feature_names[i]]),
        }
        for i in top_indices
    ]

    return {
        'predicted_class': pred_class,
        'base_value': base_value,
        'top_features': top_features,
    }


# ── Test on one sample row ────────────────────────────────────────
test_row = X_sample.iloc[[0]]
result = get_shap_for_prediction(explainer, test_row, FEATURE_COLS, top_n=5)

print(f"Predicted class: {result['predicted_class']} ({TIER_NAMES[result['predicted_class']]})")
print(f"Base value: {result['base_value']:.4f}")
print("\nTop 5 features driving this prediction:")
for f in result['top_features']:
    direction = '+' if f['shap_value'] > 0 else '-'
    print(f"  {direction} {f['feature']:20s}  SHAP={f['shap_value']:+.4f}  value={f['feature_value']:.2f}")

## CELL 9 — Per-Class SHAP Breakdown (All 4 Tiers)

For a single prediction, show how each class score is decomposed.
Useful for the frontend probability + SHAP chart.

In [ ]:
def get_full_shap_breakdown(
    explainer: shap.TreeExplainer,
    row: pd.DataFrame,
    feature_names: list[str],
    top_n: int = 5,
) -> dict:
    """
    Full SHAP breakdown for all 4 classes.
    Returns probabilities + top features per class.
    This is the complete payload FastAPI will return.
    """
    shap_vals = explainer.shap_values(row)
    probabilities = classifier.predict_proba(row)[0]  # (4,)
    pred_class = int(np.argmax(probabilities))

    classes = []
    for class_idx, tier_name in enumerate(TIER_NAMES):
        if isinstance(shap_vals, list):
            class_shap = shap_vals[class_idx][0]
            base_val = float(explainer.expected_value[class_idx])
        else:
            class_shap = shap_vals[0]
            base_val = float(explainer.expected_value)

        top_indices = np.argsort(np.abs(class_shap))[::-1][:top_n]
        top_features = [
            {
                'feature': feature_names[i],
                'shap_value': float(class_shap[i]),
                'feature_value': float(row.iloc[0][feature_names[i]]),
            }
            for i in top_indices
        ]

        classes.append({
            'class_idx': class_idx,
            'class_name': tier_name,
            'probability': float(probabilities[class_idx]),
            'base_value': base_val,
            'top_features': top_features,
        })

    return {
        'predicted_class': pred_class,
        'predicted_tier': TIER_NAMES[pred_class],
        'confidence': float(probabilities[pred_class]),
        'classes': classes,
    }


# Test
breakdown = get_full_shap_breakdown(explainer, test_row, FEATURE_COLS)
print(f"Predicted: Tier {breakdown['predicted_class']} — {breakdown['predicted_tier']}")
print(f"Confidence: {breakdown['confidence']:.2%}")
print("\nAll class probabilities:")
for c in breakdown['classes']:
    print(f"  {c['class_name']:8s}: {c['probability']:.2%}")

print(f"\nTop 5 SHAP drivers for predicted class ({breakdown['predicted_tier']}):")
for f in breakdown['classes'][breakdown['predicted_class']]['top_features']:
    direction = '+' if f['shap_value'] > 0 else '-'
    print(f"  {direction} {f['feature']:20s}  SHAP={f['shap_value']:+.4f}")

## CELL 10 — Waterfall Plot (Single Prediction)

Visual SHAP waterfall for predicted class. This is what the React chart replicates.

In [ ]:
pred_class = breakdown['predicted_class']

if isinstance(shap_vals, list):
    single_shap = shap_vals[pred_class][0]
    base_val = explainer.expected_value[pred_class]
else:
    single_shap = shap_vals[0]
    base_val = explainer.expected_value

# Build Explanation object for waterfall
explanation = shap.Explanation(
    values=single_shap,
    base_values=base_val,
    data=test_row.values[0],
    feature_names=FEATURE_COLS,
)

plt.figure()
shap.plots.waterfall(explanation, show=False)
plt.title(f'SHAP Waterfall — Predicted: {TIER_NAMES[pred_class]}')
plt.tight_layout()
plt.savefig('models/shap_waterfall_example.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: models/shap_waterfall_example.png")

## CELL 11 — Save Explainer + Verify

Save the explainer for FastAPI. Verify it loads and works correctly.

In [ ]:
EXPLAINER_PATH = 'models/shap_explainer.pkl'

joblib.dump(explainer, EXPLAINER_PATH)
print(f"Explainer saved: {EXPLAINER_PATH}")
print(f"File size: {os.path.getsize(EXPLAINER_PATH) / 1024 / 1024:.1f} MB")

# ── Verify round-trip ─────────────────────────────────────────────
print("\nVerifying explainer round-trip...")
loaded_explainer = joblib.load(EXPLAINER_PATH)

# Must return same result as original
verify_shap = loaded_explainer.shap_values(test_row)
if isinstance(verify_shap, list):
    original = explainer.shap_values(test_row)
    match = np.allclose(verify_shap[0], original[0])
    print(f"SHAP values match after reload: {match}")
print("Explainer verified OK")

## CELL 12 — Download All SHAP Artifacts

In [ ]:
import shutil
from google.colab import files

# Zip everything in models/ (includes shap_explainer.pkl + plots)
shutil.make_archive('shap_artifacts', 'zip', 'models/')
files.download('shap_artifacts.zip')
print("Downloaded: shap_artifacts.zip")
print("Extract to api/models/ locally")

## CELL 13 — API Integration Snippet

Copy-paste this into `api/main.py`.
This is the exact function FastAPI will call on `/predict`.

In [ ]:
api_snippet = '''
# ── In api/main.py ────────────────────────────────────────────────

import joblib
import numpy as np
import pandas as pd
import shap

TIER_NAMES = ["None", "Low", "Medium", "High"]
FEATURE_COLS = [
    "STATE", "MONTH_NAME", "EVENT_TYPE",
    "INJURIES_DIRECT", "INJURIES_INDIRECT",
    "DEATHS_DIRECT", "DEATHS_INDIRECT",
    "MAGNITUDE", "duration_min",
    "season", "state_avg_damage", "event_avg_damage",
]

# Load once at startup
classifier    = joblib.load("models/classifier.pkl")
shap_explainer = joblib.load("models/shap_explainer.pkl")
le_state      = joblib.load("models/le_state.pkl")
le_event_type = joblib.load("models/le_event_type.pkl")
le_month_name = joblib.load("models/le_month_name.pkl")


def compute_shap_explanation(row: pd.DataFrame, top_n: int = 5) -> dict:
    shap_vals = shap_explainer.shap_values(row)
    probabilities = classifier.predict_proba(row)[0]
    pred_class = int(np.argmax(probabilities))

    if isinstance(shap_vals, list):
        class_shap = shap_vals[pred_class][0]
        base_value = float(shap_explainer.expected_value[pred_class])
    else:
        class_shap = shap_vals[0]
        base_value = float(shap_explainer.expected_value)

    top_indices = np.argsort(np.abs(class_shap))[::-1][:top_n]
    top_features = [
        {
            "feature": FEATURE_COLS[i],
            "shap_value": float(class_shap[i]),
            "feature_value": float(row.iloc[0][FEATURE_COLS[i]]),
        }
        for i in top_indices
    ]

    return {
        "predicted_class": pred_class,
        "predicted_tier": TIER_NAMES[pred_class],
        "confidence": float(probabilities[pred_class]),
        "probabilities": {
            TIER_NAMES[i]: float(p) for i, p in enumerate(probabilities)
        },
        "shap": {
            "base_value": base_value,
            "top_features": top_features,
        },
    }
'''

print(api_snippet)

## CELL 14 — Quick Sanity Check Summary

In [ ]:
print("=" * 50)
print("SHAP NOTEBOOK SUMMARY")
print("=" * 50)
print(f"Explainer type  : TreeExplainer (exact, fast)")
print(f"Base values     : {explainer.expected_value}")
print(f"Sample used     : {X_sample.shape[0]} rows (stratified)")
print(f"Features        : {len(FEATURE_COLS)}")
print(f"Classes         : {TIER_NAMES}")
print()
print("Files saved:")
for fname in os.listdir('models'):
    path = f'models/{fname}'
    size_kb = os.path.getsize(path) / 1024
    print(f"  {fname:40s}  {size_kb:.0f} KB")
print()
print("Next step: Build api/main.py with FastAPI + /predict endpoint")